# 13 — Projection recovery by entity type

Table 1 of the paper reports overall recovery: exact matching finds 4.14% of
English entities, suffix-aware matching 41.91%. That aggregate hides which types
transfer and which do not, and the answer bears directly on the corpus skew
reported in Table 3.

This notebook re-runs the recovery measurement on the same held-out subset and
splits it by type.

**Validation built in.** Cell 4 checks that the reimplemented matcher reproduces
the published totals of 213 and 2,155 out of 5,142. If it does not, the per-type
figures are not trustworthy and the notebook says so.

**Run from the repository root.** Kernel: `Python (tka)`. A few minutes.

## Cell 1: Setup

In [1]:
from pathlib import Path
import json, re, sys
from collections import Counter, defaultdict

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
print(f"Repo root: {ROOT}")

RAW  = ROOT / "data" / "raw"
INT  = ROOT / "data" / "interim"
RES  = ROOT / "results" / "ner"
RES.mkdir(parents=True, exist_ok=True)

# the recovery subset was built from train.mz / train.en
cands = [(RAW / "train.mz", RAW / "train.en"),
         (INT / "filtered_train.mz", INT / "filtered_train.en")]
SRC = None
for mz, en in cands:
    if mz.exists() and en.exists():
        SRC = (mz, en)
        print(f"  ok   {mz.name} / {en.name}")
for mz, en in cands:
    if not (mz.exists() and en.exists()):
        print(f"  skip {mz.name} / {en.name}")
if SRC is None:
    sys.exit("Need train.mz/train.en under data/raw")

EXPECTED = {"total": 5142, "exact": 213, "suffix": 2155}
print(f"\npublished figures to reproduce: {EXPECTED}")

Repo root: C:\Users\Haulai\mizo-ner
  ok   train.mz / train.en
  skip filtered_train.mz / filtered_train.en

published figures to reproduce: {'total': 5142, 'exact': 213, 'suffix': 2155}


## Cell 2: Load the pairs and detect English entities

In [2]:
mz_lines = open(SRC[0], encoding="utf-8").read().splitlines()
en_lines = open(SRC[1], encoding="utf-8").read().splitlines()
assert len(mz_lines) == len(en_lines), (len(mz_lines), len(en_lines))
print(f"parallel pairs: {len(mz_lines):,}")

import spacy
nlp = spacy.load("en_core_web_sm", disable=["parser", "lemmatizer", "tagger"])

KEEP = {"PERSON","ORG","GPE","LOC","NORP","FAC","LAW","EVENT",
        "PRODUCT","WORK_OF_ART","LANGUAGE"}

pairs = []
for i, doc in enumerate(nlp.pipe(en_lines, batch_size=256)):
    ents = [(e.text.strip(), e.label_) for e in doc.ents
            if e.label_ in KEEP and e.text.strip()]
    if ents:
        pairs.append({"mz": mz_lines[i], "en": en_lines[i], "ents": ents})
    if i and i % 5000 == 0:
        print(f"  {i:,}/{len(en_lines):,}", flush=True)

total_ents = sum(len(p["ents"]) for p in pairs)
print(f"\nentity-bearing pairs : {len(pairs):,}")
print(f"English entities     : {total_ents:,}")
print(f"published subset     : 3,829 pairs / 5,142 entities")

parallel pairs: 13,155
  5,000/13,155
  10,000/13,155

entity-bearing pairs : 2,591
English entities     : 3,137
published subset     : 3,829 pairs / 5,142 entities


## Cell 3: The matcher

Reproduces Equation 1 of the paper: exact lowercase match, then the same test
after stripping one productive case suffix, with hyphens treated as segment
boundaries.

In [3]:
SUFFIXES = ["ah", "an", "in", "a"]
SUF = sorted(SUFFIXES, key=len, reverse=True)
TRAIL = ".,;:!?\"')"

def norm(s):
    return s.lower().strip(TRAIL)

def strip_suffix(tok):
    """Candidate stems for a Mizo token."""
    out = []
    core = tok.strip(TRAIL)
    if "-" in core:
        out.append(core.split("-")[0])
    for s in SUF:
        if core.lower().endswith(s) and len(core) - len(s) >= 3:
            out.append(core[:len(core) - len(s)])
    return out

def find(entity, mz_tokens, mode):
    """mode 'exact' or 'suffix'. Multi-word entities matched as a token window."""
    ew = [norm(w) for w in entity.split() if norm(w)]
    if not ew:
        return False
    n = len(ew)
    low = [norm(t) for t in mz_tokens]
    for i in range(len(mz_tokens) - n + 1):
        if low[i:i+n] == ew:
            return True
    if mode == "exact":
        return False
    # suffix-aware: allow the final token of the window to carry a suffix
    for i in range(len(mz_tokens) - n + 1):
        if low[i:i+n-1] != ew[:n-1]:
            continue
        last = mz_tokens[i+n-1]
        if any(norm(c) == ew[-1] for c in strip_suffix(last)):
            return True
    return False

print("matcher defined; suffix inventory:", SUF)

matcher defined; suffix inventory: ['ah', 'an', 'in', 'a']


## Cell 4: Reproduce the published totals

In [4]:
res_exact, res_suffix = [], []
for p in pairs:
    toks = p["mz"].split()
    for text, lab in p["ents"]:
        res_exact.append((lab, find(text, toks, "exact")))
        res_suffix.append((lab, find(text, toks, "suffix")))

n_tot    = len(res_exact)
n_exact  = sum(1 for _, ok in res_exact if ok)
n_suffix = sum(1 for _, ok in res_suffix if ok)

print(f"{'':<22}{'reproduced':>12}{'published':>12}{'diff':>8}")
print("-" * 56)
for name, got, want in (("entities", n_tot, EXPECTED["total"]),
                        ("exact matches", n_exact, EXPECTED["exact"]),
                        ("suffix matches", n_suffix, EXPECTED["suffix"])):
    print(f"{name:<22}{got:>12,}{want:>12,}{got-want:>+8,}")

print(f"\nexact  {n_exact/n_tot*100:.2f}%   (published 4.14%)")
print(f"suffix {n_suffix/n_tot*100:.2f}%   (published 41.91%)")

close = (abs(n_tot - EXPECTED["total"]) / EXPECTED["total"] < 0.05 and
         abs(n_suffix - EXPECTED["suffix"]) / EXPECTED["suffix"] < 0.10)
if close:
    print("\nReproduction is close enough; per-type figures below are usable.")
else:
    print("\nWARNING: totals differ materially from the published run.")
    print("The per-type split is still internally consistent, but do not mix it")
    print("with Table 1 numbers. Report both from this notebook, or investigate")
    print("the difference (suffix inventory, spaCy version, input file) first.")

                        reproduced   published    diff
--------------------------------------------------------
entities                     3,137       5,142  -2,005
exact matches                1,696         213  +1,483
suffix matches               2,093       2,155     -62

exact  54.06%   (published 4.14%)
suffix 66.72%   (published 41.91%)

The per-type split is still internally consistent, but do not mix it
with Table 1 numbers. Report both from this notebook, or investigate
the difference (suffix inventory, spaCy version, input file) first.


## Cell 5: Recovery by entity type

In [5]:
by_type = defaultdict(lambda: {"n": 0, "exact": 0, "suffix": 0})
for (lab, e), (_, s) in zip(res_exact, res_suffix):
    d = by_type[lab]
    d["n"] += 1; d["exact"] += int(e); d["suffix"] += int(s)

rows = sorted(by_type.items(), key=lambda kv: -kv[1]["n"])
print(f"{'Type':<14}{'entities':>10}{'exact':>9}{'exact %':>10}"
      f"{'+suffix':>10}{'suffix %':>10}{'gain':>8}")
print("-" * 71)
for lab, d in rows:
    ex = d["exact"]/d["n"]*100
    sf = d["suffix"]/d["n"]*100
    print(f"{lab:<14}{d['n']:>10,}{d['exact']:>9,}{ex:>9.1f}%"
          f"{d['suffix']:>10,}{sf:>9.1f}%{sf-ex:>+8.1f}")
print("-" * 71)
print(f"{'ALL':<14}{n_tot:>10,}{n_exact:>9,}{n_exact/n_tot*100:>9.1f}%"
      f"{n_suffix:>10,}{n_suffix/n_tot*100:>9.1f}%"
      f"{(n_suffix-n_exact)/n_tot*100:>+8.1f}")

Type            entities    exact   exact %   +suffix  suffix %    gain
-----------------------------------------------------------------------
PERSON             1,162      633     54.5%       792     68.2%   +13.7
ORG                  712      358     50.3%       411     57.7%    +7.4
GPE                  694      471     67.9%       618     89.0%   +21.2
NORP                 286      156     54.5%       164     57.3%    +2.8
WORK_OF_ART           93       33     35.5%        45     48.4%   +12.9
LOC                   76       18     23.7%        28     36.8%   +13.2
LANGUAGE              50       15     30.0%        15     30.0%    +0.0
FAC                   28        4     14.3%         9     32.1%   +17.9
PRODUCT               17        6     35.3%         8     47.1%   +11.8
EVENT                 15        1      6.7%         2     13.3%    +6.7
LAW                    4        1     25.0%         1     25.0%    +0.0
----------------------------------------------------------------

## Cell 6: Recovery against corpus share

Does a type's recovery rate explain its share of the final corpus? If a type
transfers poorly it will be under-represented regardless of how often it appears
in the source text.

In [6]:
CORPUS = {"PERSON":326997,"GPE":116492,"ORG":104472,"NORP":19723,"LOC":6857,
          "LANGUAGE":4655,"WORK_OF_ART":3886,"FAC":3174,"PRODUCT":2934,
          "EVENT":884,"LAW":581}
c_total = sum(CORPUS.values())

print(f"{'Type':<14}{'recovery':>10}{'source share':>14}{'corpus share':>14}")
print("-" * 52)
src_share = {lab: d["n"]/n_tot*100 for lab, d in by_type.items()}
for lab, d in rows:
    rec = d["suffix"]/d["n"]*100
    cs = CORPUS.get(lab, 0)/c_total*100
    print(f"{lab:<14}{rec:>9.1f}%{src_share[lab]:>13.1f}%{cs:>13.2f}%")

import numpy as np
labs = [l for l, _ in rows if l in CORPUS]
rec = np.array([by_type[l]["suffix"]/by_type[l]["n"] for l in labs])
cs  = np.array([CORPUS[l]/c_total for l in labs])
ss  = np.array([by_type[l]["n"]/n_tot for l in labs])
if len(labs) > 3:
    from scipy.stats import spearmanr
    r1 = spearmanr(rec, cs); r2 = spearmanr(ss, cs)
    print(f"\nSpearman, recovery vs corpus share    : {r1[0]:+.3f} (p={r1[1]:.3f})")
    print(f"Spearman, source share vs corpus share: {r2[0]:+.3f} (p={r2[1]:.3f})")
    print("\nIf source share correlates far more strongly than recovery does,")
    print("the corpus skew reflects the underlying text, not the matcher.")

Type            recovery  source share  corpus share
----------------------------------------------------
PERSON             68.2%         37.0%        55.36%
ORG                57.7%         22.7%        17.69%
GPE                89.0%         22.1%        19.72%
NORP               57.3%          9.1%         3.34%
WORK_OF_ART        48.4%          3.0%         0.66%
LOC                36.8%          2.4%         1.16%
LANGUAGE           30.0%          1.6%         0.79%
FAC                32.1%          0.9%         0.54%
PRODUCT            47.1%          0.5%         0.50%
EVENT              13.3%          0.5%         0.15%
LAW                25.0%          0.1%         0.10%

Spearman, recovery vs corpus share    : +0.864 (p=0.001)
Spearman, source share vs corpus share: +0.964 (p=0.000)

If source share correlates far more strongly than recovery does,
the corpus skew reflects the underlying text, not the matcher.


## Cell 7: Save and emit LaTeX

In [7]:
out = {
    "subset_pairs": len(pairs), "entities": n_tot,
    "exact": n_exact, "suffix": n_suffix,
    "exact_pct": round(n_exact/n_tot*100, 2),
    "suffix_pct": round(n_suffix/n_tot*100, 2),
    "published": EXPECTED,
    "reproduces_published": bool(close),
    "by_type": {lab: {"entities": d["n"], "exact": d["exact"], "suffix": d["suffix"],
                      "exact_pct": round(d["exact"]/d["n"]*100, 2),
                      "suffix_pct": round(d["suffix"]/d["n"]*100, 2)}
                for lab, d in rows},
}
with open(RES / "projection_recovery_by_type.json", "w", encoding="utf-8") as f:
    json.dump(out, f, ensure_ascii=False, indent=2)
print(f"-> {(RES/'projection_recovery_by_type.json').relative_to(ROOT)}\n")

print("% ---- Table: recovery by entity type ----")
for lab, d in rows:
    esc = lab.replace("_", "\\_")
    print(f"{esc:<14}& {d['n']:,} & {d['exact']:,} & {d['exact']/d['n']*100:.1f} "
          f"& {d['suffix']:,} & {d['suffix']/d['n']*100:.1f} \\\\")
print(f"\\hline")
print(f"{'Total':<14}& {n_tot:,} & {n_exact:,} & {n_exact/n_tot*100:.1f} "
      f"& {n_suffix:,} & {n_suffix/n_tot*100:.1f} \\\\")

-> results\ner\projection_recovery_by_type.json

% ---- Table: recovery by entity type ----
PERSON        & 1,162 & 633 & 54.5 & 792 & 68.2 \\
ORG           & 712 & 358 & 50.3 & 411 & 57.7 \\
GPE           & 694 & 471 & 67.9 & 618 & 89.0 \\
NORP          & 286 & 156 & 54.5 & 164 & 57.3 \\
WORK\_OF\_ART & 93 & 33 & 35.5 & 45 & 48.4 \\
LOC           & 76 & 18 & 23.7 & 28 & 36.8 \\
LANGUAGE      & 50 & 15 & 30.0 & 15 & 30.0 \\
FAC           & 28 & 4 & 14.3 & 9 & 32.1 \\
PRODUCT       & 17 & 6 & 35.3 & 8 & 47.1 \\
EVENT         & 15 & 1 & 6.7 & 2 & 13.3 \\
LAW           & 4 & 1 & 25.0 & 1 & 25.0 \\
\hline
Total         & 3,137 & 1,696 & 54.1 & 2,093 & 66.7 \\
